# Bi-Int Digital Twin — Evaluation Notebook

**Last updated:** 21 May 2026  
**Environment:** TwinCell (conda), Python 3.11, TF 2.15.0, GPU RTX 4000 Ada 20 475 MiB

## Contents
1. Setup & imports
2. CCLE dataset summary (confirmed dimensions)
3. ChEMBL GNN pre-training curves (real results — session 21/05/2026)
4. BiInt QSAR training curve (pending — run in progress at batch_size=16)
5. Drug SMILES mapping summary
6. DQN BRICS version — final results
7. Baseline comparison (run `python baseline_models.py` first)
8. Current limitations & next steps

---
### P1–P7 Fixes applied (21 May 2026)

| Fix | Description | Status |
|-----|-------------|--------|
| P1 | BRICSMolecularFeaturizer: real topology adjacency + 3-level SMILES lookup | Done |
| P2 | Mutation alignment: full `Tumor_Sample_Barcode` match; sorted cell order; shape assertions | Done |
| P3 | IC50 validation diagnostics; 3 split modes (random / leave_drug_out / leave_cell_out) | Done |
| P4 | DQN reward: SA score + Lipinski hard penalty (−2.0) + Tanimoto CCLE diversity | Done |
| P5 | baseline_models.py: Ridge/RF/MLP/XGBoost, ECFP4(2048)+GEx+CNA+mut, R² metric | Done |
| P6 | TensorBoard + CSVLogger + EarlyStopping + gradient norm in BiIntTrainer | Done |
| P7 | OOM fix: float32 downcast + del DataFrames + NPZ omics cache + batch_size 32→16 | Done |


In [ ]:
# Cell 1 — Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
sys.path.insert(0, '..')

try:
    from rdkit import Chem
    from rdkit.Chem import QED, Descriptors
    RDKIT_OK = True
except ImportError:
    print('RDKit not available — some cells will be skipped')
    RDKIT_OK = False

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
print('Setup OK')

## 2. CCLE Dataset Summary
Confirmed dimensions from the run of 21 May 2026 (after P1/P2/P7 fixes).

In [ ]:
# Cell 2 — CCLE dataset confirmed dimensions
ccle_summary = {
    'Drugs (total IC50 matrix)': 266,
    'Cell lines (total IC50 matrix)': 1068,
    'Common cell lines (IC50 + GEx + CNA)': 647,
    'Drugs with valid SMILES (PubChem)': 201,
    'Drugs missing SMILES': 65,
    'Valid triplets (drug, cell, IC50)': 103477,
    'GEx shape (cells x genes)': '(647, 978)',
    'CNA shape (cells x genes)': '(647, 426)',
    'Mutations shape (cells x genes)': '(647, 735)',
    'Mutation sparsity': '0.844',
    'Mean mutations per cell': '115.0',
    'IC50 range (uM)': '0.0001 — 400374.7',
    'IC50 post-log1p mean': '2.6747',
    'IC50 post-log1p std': '1.8512',
    'Outliers >100 uM (kept)': '23163 (16.9%)',
    'NPZ cache': 'Dataset/ccle_broad_2019/omics_cache_gex978_cna426.npz',
}

df_ccle = pd.DataFrame(list(ccle_summary.items()), columns=['Metric', 'Value'])
display(df_ccle.to_string(index=False))

# Visual: pie chart of triplet coverage
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# SMILES coverage
axes[0].pie([201, 65], labels=['SMILES found\n(201)', 'Missing\n(65)'],
            colors=['#4CAF50', '#F44336'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Drug SMILES coverage\n(266 total CCLE drugs)')

# Omics modalities
axes[1].bar(['GEx genes', 'CNA genes', 'Mut genes'], [978, 426, 735],
            color=['#2196F3', '#FF9800', '#9C27B0'], edgecolor='k', linewidth=0.6)
axes[1].set_ylabel('Feature dimensions')
axes[1].set_title('Omics feature dimensions per cell line\n(647 common cell lines)')
for i, v in enumerate([978, 426, 735]):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 3. ChEMBL GNN Pre-training
Real results from session 21 May 2026. 10 epochs on 100k ChEMBL molecules,
8-target multi-task regression (LogP, TPSA, MW, QED, HBD, HBA, NumRings, NumAromRings).

In [ ]:
# Cell 3 — ChEMBL pre-training REAL results (21 May 2026)
epochs     = list(range(1, 11))
train_rmse = [0.4886, 0.3462, 0.3034, 0.2796, 0.2649, 0.2511, 0.2410, 0.2322, 0.2247, 0.2192]
val_rmse   = [0.3598, 0.2972, 0.2671, 0.2471, 0.2467, 0.2327, 0.2323, 0.2300, 0.2187, 0.2215]
val_mae    = [0.2537, 0.2094, 0.1871, 0.1749, 0.1733, 0.1659, 0.1672, 0.1654, 0.1552, 0.1609]
val_loss   = [0.12948, 0.08831, 0.07136, 0.06104, 0.06084, 0.05415, 0.05397, 0.05291, 0.04784, None]

best_epoch = int(np.argmin(val_rmse)) + 1  # epoch 9
best_val   = min(val_rmse)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# RMSE curves
axes[0].plot(epochs, train_rmse, 'o-', label='Train RMSE', color='#2196F3')
axes[0].plot(epochs, val_rmse,   's-', label='Val RMSE',   color='#F44336')
axes[0].plot(epochs, val_mae,    '^--', label='Val MAE',   color='#FF9800', alpha=0.7)
axes[0].annotate(
    f'Best val RMSE={best_val:.4f}\n(epoch {best_epoch})',
    xy=(best_epoch, best_val),
    xytext=(best_epoch - 3.5, best_val + 0.04),
    arrowprops=dict(arrowstyle='->', color='black'),
    fontsize=9
)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('RMSE / MAE (normalised targets)')
axes[0].set_title('ChEMBL GNN pre-training — learning curves\n(21 May 2026, GPU RTX 4000 Ada)')
axes[0].legend()
axes[0].set_xticks(epochs)
axes[0].grid(alpha=0.3)

# Val loss (early stopping criterion)
vl_clean = [v for v in val_loss if v is not None]
ep_clean = epochs[:len(vl_clean)]
axes[1].plot(ep_clean, vl_clean, 'o-', color='#9C27B0', label='Val loss (MSE)')
axes[1].axvline(9, color='green', linestyle='--', linewidth=1, label='Best checkpoint (ep 9)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val loss (MSE, 8 targets)')
axes[1].set_title('Val loss — ModelCheckpoint saved at ep 9\n(val_loss=0.04784)')
axes[1].legend()
axes[1].set_xticks(ep_clean)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

df_pretrain = pd.DataFrame({
    'epoch': epochs, 'train_rmse': train_rmse,
    'val_rmse': val_rmse, 'val_mae': val_mae
})
df_pretrain['best'] = df_pretrain['epoch'] == best_epoch
display(df_pretrain)

print(f'\nInterpretation:')
print(f'  RMSE on normalised targets: 0.219 = avg error of 0.22 sigma across 8 descriptors')
print(f'  RMSE=1.0 would mean predicting the mean (no learning)')
print(f'  Transferred layers: node_embed, gcn_proj_1, ln1, node_proj, ln2')
print(f'  Weights saved: pretrained_weights/chembl_drug_encoder.weights.h5')

## 4. BiInt QSAR Training on CCLE

**Status: run in progress (PID 143145, batch_size=8, cross_entropy, GPU RTX 4000 Ada)**

| Epoch | Train RMSE | Val RMSE | Pearson r | Grad norm |
|-------|-----------|---------|-----------|-----------|
| 1 | 0.9445 | 0.8464 | **0.492** | 28.76 |
| 2–5 | *in progress…* | | | |

**Epoch 1 interpretation:** Pearson r = 0.49 after a single epoch — the model is already learning drug-omics interaction signal. Val RMSE = 0.846 on normalised log-IC50 targets.

Previous result (before P1/P2 fixes — **INVALIDATED**):
- r = 0.884 obtained with **random drug vectors** (P1 bug) + **zero mutation matrix** (P2 bug)
- Leave-drug-out: r = −0.35 — expected with zero SAR signal

In [ ]:
# Cell 4 — BiInt QSAR training: load from CSV log if available, else show placeholder
log_path = '../logs/run_gpu_main/training_log.csv'

if os.path.exists(log_path) and os.path.getsize(log_path) > 10:
    df_qsar = pd.read_csv(log_path)
    print(f'Loaded {len(df_qsar)} epochs from {log_path}')
    display(df_qsar)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(df_qsar['epoch'], df_qsar['train_rmse'], 'o-', label='Train RMSE', color='#2196F3')
    axes[0].plot(df_qsar['epoch'], df_qsar['val_rmse'],   's-', label='Val RMSE',   color='#F44336')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('RMSE (normalised log uM)')
    axes[0].set_title('BiInt QSAR — RMSE learning curve\n(CCLE, 103 477 triplets, batch=16, GPU)')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if 'pearson_r' in df_qsar.columns:
        axes[1].plot(df_qsar['epoch'], df_qsar['pearson_r'], 'o-', color='#4CAF50', label='Val Pearson r')
        axes[1].axhline(0, color='k', linestyle='--', linewidth=0.8)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Pearson r')
        axes[1].set_title('Val Pearson r per epoch\n(>0.5 = competitive with published QSAR)')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    best = df_qsar.loc[df_qsar['val_rmse'].idxmin()]
    print(f'\nBest epoch: {int(best["epoch"])}  val_rmse={best["val_rmse"]:.4f}', end='')
    if 'pearson_r' in df_qsar.columns:
        print(f'  pearson_r={best["pearson_r"]:.4f}', end='')
    print()
else:
    print('training_log.csv not yet available — run in progress.')
    print('Command: bash ~/Twin/run_pipeline.sh')
    print()
    print('Expected output location:', log_path)
    print('Run this cell again after the pipeline completes.')

In [ ]:
# Cell 5 — Drug SMILES mapping summary
smiles_path = '../Dataset/ccle_drug_smiles.csv'

if not os.path.exists(smiles_path):
    print(f'ccle_drug_smiles.csv not found. Run: python fetch_drug_smiles.py')
else:
    df_smi = pd.read_csv(smiles_path)
    n_total  = len(df_smi)
    n_mapped = df_smi['smiles'].notna().sum()
    n_missed = n_total - n_mapped

    print(f'Drug SMILES mapping  ({smiles_path})')
    print(f'  Total  : {n_total}')
    print(f'  Mapped : {n_mapped} ({100*n_mapped/n_total:.1f}%)')
    print(f'  Missing: {n_missed} ({100*n_missed/n_total:.1f}%)')

    if 'source' in df_smi.columns:
        print('\nSource breakdown:')
        print(df_smi['source'].value_counts(dropna=False).to_string())

    mapped = df_smi[df_smi['smiles'].notna()]
    print('\n5 example mappings:')
    for _, row in mapped.head(5).iterrows():
        smi = str(row['smiles'])[:60] + ('...' if len(str(row['smiles'])) > 60 else '')
        print(f"  {row['drug_name']:<35s} -> {smi}")

    missed_list = df_smi[df_smi['smiles'].isna()]['drug_name'].tolist()
    if missed_list:
        print(f'\nUnmapped drugs ({len(missed_list)}):')
        for m in missed_list[:20]:
            print(f'  - {m}')
        if len(missed_list) > 20:
            print(f'  ... and {len(missed_list)-20} more')

## 6. BRICS-DQN — Final Results (completed run)

In [ ]:
# Cell 6 — BRICS-DQN confirmed results (from logs_brics_dqn.txt, 21 May 2026)
brics_results = {
    'Episodes': 5000,
    'Final epsilon': 0.150,
    'Validity (final)': '60.5% (3023/5000)',
    'Best reward': 6.1239,
    'Best SMILES': 'Cc1ccc(C)n1Nc1ccc(CCCCCc2csc(-c3nc(-c4ccc(Nn5c(C)ccc5C)nn4)cs3)n2)nn1',
}

top5 = [
    (6.124, 'Cc1ccc(C)n1Nc1ccc(CCCCCc2csc(-c3nc(-c4ccc(Nn5c(C)ccc5C)nn4)cs3)n2)nn1'),
    (6.078, 'NS(=O)(=O)c1ccc(-c2cccc(O)c2)cc1'),
    (6.047, 'O=C(O)CCCN1CCC(=C2CCN(CCCC(=O)O)CC2)CC1'),
    (6.041, 'c1ncc(-c2cncs2)s1'),
    (5.991, 'CCCCCC=CCCCCC'),
]

print('BRICS-DQN Final Results (5000 episodes, P4 reward: SA+Lipinski+Tanimoto)')
print('=' * 70)
for k, v in brics_results.items():
    print(f'  {k:<25s}: {v}')
print()
print('Top 5 molecules:')
for rank, (r, smi) in enumerate(top5, 1):
    print(f'  {rank}. R={r:.3f}  {smi}')

# Analyse drug-likeness of top 5 with RDKit
if RDKIT_OK:
    from rdkit.Chem import Descriptors
    print('\nDrug-likeness analysis (top 5):')
    print(f"  {'Rank':<5} {'MW':>7} {'LogP':>6} {'QED':>6} {'HBD':>5} {'HBA':>5} {'Lipinski'}")
    print('  ' + '-'*55)
    for rank, (r, smi) in enumerate(top5, 1):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mw   = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            qed  = QED.qed(mol)
            hbd  = Descriptors.NumHDonors(mol)
            hba  = Descriptors.NumHAcceptors(mol)
            lip  = 'PASS' if mw<=500 and logp<=5 and hbd<=5 and hba<=10 else 'FAIL'
            print(f'  {rank:<5} {mw:>7.1f} {logp:>6.2f} {qed:>6.3f} {hbd:>5} {hba:>5}  {lip}')
        else:
            print(f'  {rank:<5} (invalid SMILES)')

## 7. Baseline Comparison
Run `python baseline_models.py` to generate `Dataset/baseline_results.csv`.

In [ ]:
# Cell 7 — Baseline comparison vs BiInt
baseline_path = '../Dataset/baseline_results.csv'

if not os.path.exists(baseline_path):
    print(f'baseline_results.csv not found.')
    print('Generate it with: python baseline_models.py')
    print()
    print('Expected models: Ridge (ECFP4+omics), RF (ECFP4+omics), MLP (ECFP4+omics), XGBoost (ECFP4+omics)')
    print('Features: ECFP4(2048) | GEx(978) | CNA(426) | Mut(735) = 4187 dims')
    print('Splits: Random, Leave-Drug-Out, Leave-Cell-Out')
else:
    df_bl = pd.read_csv(baseline_path)
    print(f'Loaded {len(df_bl)} rows')
    display(df_bl)

    # Load BiInt results if available
    biint_rmse = None
    biint_pearson = None
    if os.path.exists(log_path) and os.path.getsize(log_path) > 10:
        df_qsar = pd.read_csv(log_path)
        best = df_qsar.loc[df_qsar['val_rmse'].idxmin()]
        biint_rmse = float(best['val_rmse'])
        if 'pearson_r' in df_qsar.columns:
            biint_pearson = float(best['pearson_r'])

    df_random = df_bl[df_bl['Split'] == 'Random'].copy()
    biint_row = pd.DataFrame([{
        'Model': 'Bi-Int (GNN+QuatVAE+4xBiInt)',
        'Split': 'Random',
        'RMSE':  biint_rmse if biint_rmse else float('nan'),
        'R2':    float('nan'),
        'Pearson_r':  biint_pearson if biint_pearson else float('nan'),
        'Spearman_r': float('nan'),
    }])
    df_plot = pd.concat([df_random, biint_row], ignore_index=True)
    df_plot_valid = df_plot.dropna(subset=['RMSE'])

    if len(df_plot_valid) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = plt.cm.Set2(np.linspace(0, 1, len(df_plot_valid)))
        bars = ax.bar(df_plot_valid['Model'], df_plot_valid['RMSE'],
                      color=colors, edgecolor='k', linewidth=0.6)
        ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
        ax.set_ylabel('Val RMSE (normalised log uM)')
        ax.set_title('IC50 prediction — Baselines vs Bi-Int (random split)\nLower is better')
        ax.set_ylim(0, df_plot_valid['RMSE'].max() * 1.25)
        plt.xticks(rotation=20, ha='right')
        plt.tight_layout()
        plt.show()

## 8. Current State & Next Steps

### What is done

| Component | Status | Key metric |
|-----------|--------|------------|
| ChEMBL GNN pre-training | Done | Val RMSE = 0.2187, Val loss = 0.0491 (epoch 9) |
| CCLE data loading (P1/P2/P3/P7 fixed) | Done | 647 cells, 103 477 triplets, 201/266 drugs with SMILES |
| Omics NPZ cache | Done | `omics_cache_gex978_cna426.npz` — instant reload |
| GPU setup (TF 2.15 + CUDA 13) | Done | RTX 4000 Ada, 20 475 MiB |
| BRICS-DQN (P4 reward) | Done | Best R=6.124, Validity=60.5% |
| P5 baselines (Ridge/RF/MLP/XGBoost) | Implemented | Results pending (`python baseline_models.py`) |
| P6 TensorBoard + CSVLogger | Done | `logs/run_gpu_main/` |

### In progress
- **BiInt 20-epoch training** (batch_size=16, run in progress)
  - Command: `bash ~/Twin/run_pipeline.sh`
  - Results will appear in `logs/run_gpu_main/training_log.csv`

### What is missing

| Item | Priority | Command |
|------|----------|---------|
| BiInt training results (corrected, batch=16) | **HIGH** | `bash ~/Twin/run_pipeline.sh` |
| Baseline comparison table | HIGH | `python baseline_models.py` |
| Leave-drug-out run | HIGH | `python fullPipeline.py --split-mode leave_drug_out` |
| Leave-cell-out run | MEDIUM | `python fullPipeline.py --split-mode leave_cell_out` |
| 65 missing drug SMILES | MEDIUM | Manual ChEMBL/CAS lookup or alternative name matching |
| GitHub push | LOW | Needs new token (previous tokens compromised) |
| DQN with real IC50 oracle | LOW | Connect BRICS-DQN reward to BiInt predictions |

### Scientific interpretation (once BiInt results available)
- **Random split Pearson r > 0.7**: model captures genuine drug-omics interaction signal
- **Leave-drug-out Pearson r > 0.3**: model generalises to unseen drug scaffolds (true SAR learning)
- **Val RMSE < 0.5 (normalised)**: competitive with DeepDR, MOLI, TGSA on CCLE
- **Train-val gap < 0.05**: model not overfitting on 103k triplets